In [ ]:
# Imports
import astropy.units as u
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import scipy
from datetime import date

matplotlib.use("tkagg")

In [2]:
sag_a_r_s = 1.2 * 10 ** 10 * u.m  # https://academic.oup.com/mnras/article/421/2/1315/1134797?login=false
r_g = sag_a_r_s / 2
c = 2.99 * 10 ** 8 * (u.m / u.s)

In [4]:
def q_squared_to_b_squared(q, v, r_s):
    """Convert from periastron to impact parameter b"""
    gamma = 1 / np.sqrt(1 - (v / c) ** 2)
    b_squared = q ** 2 * (c / v) ** 2 * (1 / (1 - r_s / q) - 1 / gamma ** 2)
    return b_squared

In [7]:
def process_time_step_orbit(r, dt, energy_term, h):
    """Calculate changes to r and the vaporization loss"""
    dr = -1 * np.sqrt(energy_term - (1 - sag_a_r_s / r) * (c ** 2 + h ** 2 / r ** 2)) * dt
    return dr


def calc_radius_loss(distance, dt):
    dr_frag = 10 * u.m / (200 * u.s) * (distance / (10 * r_g)) ** -3 * dt
    return dr_frag


def check_if_correct_q(q, b, v):
    q = q * r_g
    total = (b ** 2 - q ** 2 * c ** 2 / v ** 2 * (1 / (1 - sag_a_r_s / q) - (1 - (v / c) ** 2))).value
    return total ** 2


def numerically_determine_q(b, v):
    """Numerically determine the closest approach"""
    """Has issues"""
    result = scipy.optimize.minimize(check_if_correct_q, np.array([8]), args=(b, v), bounds=[(4.01, None)], method="Nelder-Mead")
    return result.x

In [6]:
# Track luminosity of a full flare
progenitor_radius = 5000 * u.m
time_step = 0.1 * u.s
current_r = 30 * r_g
initial_radius = np.linspace(10, 150, 1000) * u.m
current_radius = initial_radius.copy()
rho = 100 * u.kg / (u.m ** 3)
epsilon = 0.1
num_steps = 0
b_inf = 6.7 * 10 ** 6 * r_g
v_inf = 1.8 * 10 ** 2 * u.m / u.s
energy = c ** 2 * 1 / (1 - (v_inf / c) ** 2)
angular_momentum_h = v_inf * b_inf * 1 / np.sqrt(1 - (v_inf / c) ** 2)
lost_radius = np.array([0]) * u.m
radiuses = np.array([current_r.value]) * u.m
luminosities = np.zeros((1, 1000))
numerical_q = numerically_determine_q(b_inf, v_inf)[0] * r_g
predicted_luminosity = epsilon * (4/3) * np.pi * (100 * u.kg * u.m ** -3) * progenitor_radius ** 3 * c ** 2 / (200 * u.s) * (10 * r_g / numerical_q) ** 3 * (1 - sag_a_r_s / numerical_q)

# Integrate the orbit to find distance at each t
while current_r > numerical_q:
    change_r = process_time_step_orbit(current_r, time_step, energy, angular_momentum_h)
    current_r += change_r
    radiuses = np.append(radiuses, current_r)
    num_steps += 1
    if num_steps % 100 == 0:
        print(num_steps, current_r / r_g)

print(predicted_luminosity.to(u.kg * u.m ** 2 * u.s ** -3))
radius_loss_at_each_t = calc_radius_loss(radiuses, time_step)
total_radius_lost = np.cumulative_sum(radius_loss_at_each_t)
radiuses_at_each_time = np.transpose(np.tile(total_radius_lost, (len(initial_radius), 1)))
radius_at_t = np.add(-1 * radiuses_at_each_time, initial_radius)
radius_at_t[radius_at_t < 0] = 0
luminosities = (epsilon * c.value ** 2) * 4 * np.pi * rho * radius_at_t ** 2 * np.transpose(np.tile(radius_loss_at_each_t, (len(initial_radius), 1)))
scale_factors = initial_radius ** -1.8
total_volume = (4/3) * np.pi * progenitor_radius ** 3
unscaled_fragment_volume = np.sum(scale_factors * (4/3) * np.pi * initial_radius ** 3)
scale_factors_volume_adjusted = scale_factors * total_volume / unscaled_fragment_volume
scaled_luminosities = luminosities * scale_factors_volume_adjusted[None, :]

4.02e+16 m 180.0 m / s
[3.6e+10] m
[3.78e+10] m
[3.42e+10] m
[3.24e+10] m
[2.88e+10] m
[2.52e+10] m
[2.52e+10] m
[2.7e+10] m
[2.52e+10] m
[2.79e+10] m
[2.88e+10] m
[2.745e+10] m
[2.7e+10] m
[2.7675e+10] m
[2.7225e+10] m
[2.75625e+10] m
[2.7675e+10] m
[2.750625e+10] m
[2.761875e+10] m
[2.7534375e+10] m
[2.750625e+10] m
[2.75484375e+10] m
[2.75203125e+10] m
[2.75414062e+10] m
[2.75484375e+10] m
[2.75378906e+10] m
[2.7534375e+10] m
[2.75396484e+10] m
[2.75414062e+10] m
[2.75387695e+10] m
[2.75378906e+10] m
[2.7539209e+10] m
[2.75396484e+10] m
[2.75389893e+10] m
[2.75387695e+10] m
[2.75390991e+10] m
[2.7539209e+10] m
[2.75390442e+10] m
[2.75389893e+10] m
[2.75390717e+10] m
[2.75390991e+10] m
[2.75390579e+10] m
[2.75390442e+10] m
[2.75390648e+10] m
[2.75390717e+10] m
[2.75390614e+10] m
[2.75390682e+10] m
[2.75390631e+10] m
[2.75390614e+10] m
[2.75390639e+10] m
[2.75390648e+10] m
[2.75390635e+10] m
[2.75390631e+10] m
[2.75390637e+10] m
[2.75390633e+10] m
[2.75390636e+10] m
[2.75390637e+10] m

In [7]:
xs = np.arange(num_steps + 1) * time_step
time_to_periastron = xs - xs[-1]
flare_lum = np.sum(scaled_luminosities, axis=1)
seconds_integration = 120
observed = np.bincount(np.arange(len(flare_lum))//seconds_integration, flare_lum)
observed_xs = np.arange(len(observed)) * seconds_integration * time_step
fig, axs = plt.subplots(1, 1)
axs.plot(time_to_periastron, flare_lum)
axs.set_xlabel("Time relative to peri-astron [s]")
axs.set_ylabel("Luminosity [W]")
# axs.set_title("Flare from r=5000m object")

# axs[1].hist(flare_lum[flare_lum > 0], 100)
# axs[1].set_xlabel("Luminosity [W] (currently unscaled by initial mass)")
# axs[1].set_ylabel("Count of time steps in luminosity bin")
# axs[1].set_yscale('log')
# plt.plot(xs, total_radius_lost)
plt.show()

In [33]:
fig, axs = plt.subplots(1, 2)
jwst_data = np.loadtxt('sgra_frame_lightcurve_all.csv', delimiter=',', skiprows=1, usecols=(4, 7, 8))
axs[0].plot(time_to_periastron, flare_lum, label="Predicted flare brightness")
axs[0].set_xlabel("Time relative to peri-astron [s]")
axs[0].set_ylabel("Luminosity [W]")
axs[0].legend()

axs[1].plot((jwst_data[:, 0] - 60405.35070) * 24 * 3600, jwst_data[:, 1], label="Observed flare")
axs[1].set_ylabel("Observed flux in 2.1$\\mu$m band [mJy]")
axs[1].set_xlabel("Time since flare peak [s]")
axs[1].set_xlim(-1000, 1000)
axs[1].legend()
plt.show()

NameError: name 'time_to_periastron' is not defined

In [8]:
def integrate_orbit(q, v_0, dt=0.1 * u.s, output=False):
    """Integrate the radius at a given time for an orbit with the given parameters."""
    steps_done = 0
    b_0 = np.sqrt(q_squared_to_b_squared(q, v_0, sag_a_r_s))
    d_0 = 30 * r_g
    orbit_energy = c ** 2 * 1 / (1 - (v_0 / c) ** 2)
    h = v_0 * b_0 * 1 / np.sqrt(1 - (v_0 / c) ** 2)
    distances = np.array([d_0.value]) * u.m
    # q = numerically_determine_q(b_0, v_0)[0] * r_g
    # if np.log10(check_if_correct_q(q / r_g, b_0, v_0)) > 50:
    #     print(f"Orbit: {b_0}, {v_0} has undefined q")
    #     return 0, 0, False
    # if q >= 30 * r_g:
    #     print(f"Orbit: {b_0}, {v_0} doesn't tidally disrupt")
    #     return 0, 0, False
    # print(check_if_correct_q(q / r_g, b_0, v_0), q / r_g)

    # Integrate the orbit to find distance at each t
    print(d_0, q)
    while d_0 > q:
        delta_r = process_time_step_orbit(d_0, dt, orbit_energy, h)
        d_0 += delta_r
        distances = np.append(distances, d_0)
        steps_done += 1
        if steps_done % 100 == 0 and output:
            print(steps_done, d_0 / r_g)

    times = np.arange(steps_done + 1) * dt
    times_since_periastron = times - times[-1]

    return distances, times_since_periastron, True

In [9]:
def simulate_store_orbit(q_0, v_0, output=False):
    """Simulate and record the integrated orbit"""
    distances, times_since_periastron, success = integrate_orbit(q_0, v_0, output=output)
    print(len(times_since_periastron))
    if not success:
        return
    orbits_completed_file = "Text_files/orbits_integrated.csv"
    integrations_file = "Text_files/integrations.csv"

    with open(orbits_completed_file, "a+") as orbits_completed:
        orbits_completed.seek(0)
        existing_orbits = orbits_completed.readlines()
        if f"{date.today()},{q_0},{v_0}\n" in existing_orbits:
            print("orbit exists")
            return

        with open(integrations_file, "a") as integrations:
            rows = [f"{date.today()},{q_0.value},{v_0.value},{t:.1f},{d:.0f}" for t, d in np.column_stack((times_since_periastron.value, distances.value))]
            text_to_write = "\n".join(rows)
            integrations.write(text_to_write + "\n")
            integrations.close()

        orbits_completed.write(f"{date.today()},{q_0},{v_0}\n")
        orbits_completed.close()


In [11]:
min_b = np.sqrt(q_squared_to_b_squared(4 * r_g, 1770 * u.m / u.s, sag_a_r_s))
max_b = np.sqrt(q_squared_to_b_squared(30 * r_g, 1770 * u.m / u.s, sag_a_r_s))
print(min_b / r_g, max_b / r_g)
simulate_store_orbit(5 * r_g, 1770 * u.m / u.s)

675706.2147011051 1354425.6161257194


In [36]:
# Generate and record many orbits
vs_to_test = np.logspace(0, 6, 10) * u.m / u.s
qs_to_test = np.linspace(4.1, 30, 10) * r_g

for current_v in vs_to_test:
    for current_q in qs_to_test:
        print(current_q / r_g, current_v)
        simulate_store_orbit(current_q, current_v, output=False)

4.1 1.0 m / s
180000000000.0 m 24599999999.999996 m
24443
6.977777777777777 1.0 m / s
180000000000.0 m 41866666666.666664 m
22305
9.855555555555554 1.0 m / s
180000000000.0 m 59133333333.33333 m
22945
12.733333333333333 1.0 m / s
180000000000.0 m 76400000000.0 m
23363
15.611111111111112 1.0 m / s
180000000000.0 m 93666666666.66667 m
23320
18.488888888888887 1.0 m / s
180000000000.0 m 110933333333.33333 m
22663
21.366666666666667 1.0 m / s
180000000000.0 m 128200000000.0 m
21201
24.244444444444447 1.0 m / s
180000000000.0 m 145466666666.6667 m
18601
27.12222222222222 1.0 m / s
180000000000.0 m 162733333333.3333 m
14067
30.0 1.0 m / s
180000000000.0 m 180000000000.0 m
1
4.1 4.641588833612778 m / s
180000000000.0 m 24599999999.999996 m
24443
6.977777777777777 4.641588833612778 m / s
180000000000.0 m 41866666666.666664 m
22305
9.855555555555554 4.641588833612778 m / s
180000000000.0 m 59133333333.33333 m
22945
12.733333333333333 4.641588833612778 m / s
180000000000.0 m 76400000000.0 m
2336

In [38]:
q, v = 5 * r_g, 1770 * u.m / u.s
dts = [1, 0.1, 0.001]
plt.figure()
for time_step in dts:
    distances, times_since_periastron, success = integrate_orbit(q, v, time_step * u.s)
    plt.plot(times_since_periastron - times_since_periastron[0], distances, label=f"{time_step}s")

plt.show()

180000000000.0 m 30000000000.0 m
180000000000.0 m 30000000000.0 m
180000000000.0 m 30000000000.0 m


In [14]:
# Read in orbits from text file
def get_specific_orbit(q_to_get, v_to_get, orbit_file):
    """Retrieve the distances and times for a given orbit, using date, q and v"""
    all_orbits = np.loadtxt(orbit_file, delimiter=",", usecols=(1, 2, 3, 4))
    correct_orbit = all_orbits[all_orbits[:, 0] == q_to_get][all_orbits[all_orbits[:, 0] == q_to_get][:, 1] == v_to_get]
    print(f"The given orbit has {len(correct_orbit[:, 0])} matching points")
    return correct_orbit[:, 2], correct_orbit[:, 3]

def get_all_orbits(orbit_file):
    """Get all orbits from orbit file"""
    all_orbits = np.loadtxt(orbit_file, delimiter=",", usecols=(1, 2, 3, 4))
    processed_orbits = []
    current_orbit = all_orbits[0, 0:2]
    current_orbit_data = np.array([0, 0, 0, 0])
    for row_index in range(len(all_orbits[:, 0])):
        if all_orbits[row_index, 0:2] != current_orbit:
            processed_orbits.append(current_orbit_data[1:, :])
            current_orbit_data = np.array([0, 0, 0, 0])
            current_orbit = all_orbits[row_index, 0:2]

        current_orbit_data = np.vstack((current_orbit_data, all_orbits[row_index, :]))

    return processed_orbits

In [16]:
print(len(get_all_orbits("./Text_files/integrations.csv")))

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()